# CLS Pipeline — Cross-Lingual Semantics Analysis

Pipeline computazionale per l'analisi della divergenza semantica tra spazi di embedding
**WEIRD** (inglese, `intfloat/multilingual-e5-large-instruct`) e **Sinico** (cinese, `BAAI/bge-large-zh-v1.5`).

## 5 Esperimenti

| # | Esperimento | Metodo | Riferimento |
|---|-------------|--------|-------------|
| 1 | **RSA + Mantel** | Correlazione tra matrici di dissimilarita' (RDM) | Kriegeskorte et al. (2008) |
| 2 | **Gromov-Wasserstein** | Distanza di trasporto ottimale tra spazi metrici | Alvarez-Melis & Jaakkola (2018) |
| 3 | **Assi Assiologici** | Proiezione su assi semantici (Kozlowski) | Kozlowski et al. (2019) |
| 4 | **Clustering Gerarchico** | Fowlkes-Mallows multi-k | Fowlkes & Mallows (1983) |
| 5 | **NDA** | Divergenza di vicinato k-NN + decomposizioni normative | Haemmerli et al. (2024) |

**Supplementare**: Visualizzazione UMAP (McInnes et al., 2018)

**Fonte dati**: HK DOJ Bilingual Legal Glossary (~394 termini core + ~324 background + 50 controllo)

In [1]:
# ── Configurazione ambiente ──────────────────────────────────────────
%matplotlib inline
%load_ext autoreload
%autoreload 2

import json
import gc
import logging
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import plotly.graph_objects as go
import pandas as pd

# Plotly in modalita' notebook per rendering inline
pio.renderers.default = "notebook"

# Filtro warning non critici (convergenza UMAP, deprecation)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_jobs.*")

# Logging minimale per il notebook
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("cls_pipeline")

print("Ambiente configurato.")

Ambiente configurato.


## Modalita' di esecuzione

Il notebook supporta due modalita':

1. **Pipeline completa** (`LOAD_FROM_RESULTS = False`): carica il dataset, genera gli embedding,
   esegue tutti e 5 gli esperimenti, produce le visualizzazioni e salva `results.json`.
   Richiede accesso alla GPU/MPS e ai modelli di embedding.

2. **Solo visualizzazione** (`LOAD_FROM_RESULTS = True`): carica un `results.json` esistente
   e salta direttamente alla ricostruzione dei risultati e ai grafici.
   Non richiede GPU ne' modelli.

Impostare il toggle nella cella successiva.

In [ ]:
# ── Parametri e configurazione ───────────────────────────────────────

# Toggle: True = carica risultati esistenti, False = esegui pipeline completa
LOAD_FROM_RESULTS = False

# Percorso al file results.json (usato se LOAD_FROM_RESULTS = True)
RESULTS_PATH = Path("output/results.json")

# Carica configurazione da config.yaml
from src.core.config_loader import load_config
config = load_config()

# ── Override opzionali per iterazione rapida (decommentare se necessario) ──
# config.experiments.rsa.n_permutations = 500
# config.experiments.gromov_wasserstein.n_permutations = 500
# config.experiments.clustering.n_permutations = 500
# config.experiments.nda.n_permutations = 500
# config.experiments.axes.n_bootstrap = 500

print(f"CLS Pipeline v{config.version}")
print(f"Modalita': {'Caricamento risultati' if LOAD_FROM_RESULTS else 'Pipeline completa'}")
print(f"Random seed: {config.random_seed}")

In [ ]:
# ── Caricamento dataset ──────────────────────────────────────────────
from src.cli import load_dataset

data = load_dataset(config)

core_terms = data["core_terms"]
background_terms = data["background_terms"]
control_terms = data.get("control_terms", [])
all_terms = core_terms + background_terms + control_terms
value_axes = data["value_axes"]
decompositions = data["normative_decompositions"]

# Liste di testo per embedding
en_core = [t["en"] for t in core_terms]
zh_core = [t["zh"] for t in core_terms]
en_all = [t["en"] for t in all_terms]
zh_all = [t["zh"] for t in all_terms]
domains_core = [t.get("domain", "unknown") for t in core_terms]

# Statistiche dataset
print(f"\nDataset caricato:")
print(f"  Core:       {len(core_terms)} termini")
print(f"  Background: {len(background_terms)} termini")
print(f"  Controllo:  {len(control_terms)} termini")
print(f"  Totale:     {len(all_terms)} termini")
print(f"  Assi:       {len(value_axes)}")
print(f"  Decomposizioni: {len(decompositions)}")
print(f"\nDomini (core): {dict(Counter(domains_core))}")

In [ ]:
# ── Branch: carica results.json OPPURE prepara pipeline ──────────────

results_data = None
experiments_data = None

if LOAD_FROM_RESULTS:
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(f"Results file non trovato: {RESULTS_PATH}")
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        results_data = json.load(f)
    experiments_data = results_data.get("experiments", {})
    meta = results_data.get("metadata", {})
    print(f"Risultati caricati da: {RESULTS_PATH}")
    print(f"  Versione: {meta.get('pipeline_version', 'N/A')}")
    print(f"  Timestamp: {meta.get('timestamp', 'N/A')}")
    print(f"  Esperimenti disponibili: {list(experiments_data.keys())}")
else:
    print("Modalita' pipeline completa: gli esperimenti verranno eseguiti nelle celle successive.")

## Sezione 2: Embedding

Estrazione dei vettori di embedding dai due modelli:

- **WEIRD** (`intfloat/multilingual-e5-large-instruct`): modello multilingue ottimizzato per inglese,
  addestrato prevalentemente su corpora occidentali. Dimensione: 1024.
- **Sinico** (`BAAI/bge-large-zh-v1.5`): modello addestrato specificamente su testi cinesi.
  Dimensione: 1024.

Gli embedding vengono cachati su disco dopo la prima estrazione.
Questa sezione viene saltata se `LOAD_FROM_RESULTS = True`.

In [ ]:
# ── Estrazione embedding (solo se pipeline completa) ─────────────────

if not LOAD_FROM_RESULTS:
    from src.core.device import DeviceManager
    from src.embeddings.client import EmbeddingClient

    device_manager = DeviceManager(config.device.preferred)
    device_info = device_manager.detect()
    print(f"Device: {device_info.device_name} ({device_info.device_type})")

    client = EmbeddingClient(config, device_manager)

    # Helper functions per embedding
    def embed_weird(texts: list[str]) -> np.ndarray:
        return client.get_embeddings(texts, model_type="weird")

    def embed_sinic(texts: list[str]) -> np.ndarray:
        return client.get_embeddings(texts, model_type="sinic")

    # Embedding dei termini core
    emb_weird_core = embed_weird(en_core)
    emb_sinic_core = embed_sinic(zh_core)
    print(f"Core embeddings — WEIRD: {emb_weird_core.shape}, Sinic: {emb_sinic_core.shape}")

    # Embedding del corpus completo (core + background + controllo)
    emb_weird_all = embed_weird(en_all)
    emb_sinic_all = embed_sinic(zh_all)
    print(f"Full corpus — WEIRD: {emb_weird_all.shape}, Sinic: {emb_sinic_all.shape}")
else:
    print("Embedding saltati (modalita' LOAD_FROM_RESULTS).")

In [ ]:
# ── Pulizia memoria GPU ──────────────────────────────────────────────
# Libera i modelli dalla memoria dopo l'estrazione degli embedding.
# Gli array numpy restano disponibili per gli esperimenti.

if not LOAD_FROM_RESULTS:
    client.unload_models()
    gc.collect()
    try:
        import torch
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    print("Modelli scaricati, memoria GPU liberata.")
else:
    print("Nessun modello da scaricare.")

## Esperimento 1: RSA + Mantel Test

**Representational Similarity Analysis** (Kriegeskorte et al., 2008):
confronta le *geometrie* dei due spazi calcolando le matrici di dissimilarita'
rappresentazionale (RDM) e la loro correlazione di Spearman.

Il **test di Mantel** (1967) valuta la significativita' statistica della
correlazione tra matrici tramite permutazione.

In [ ]:
# ── Esperimento 1: RSA ───────────────────────────────────────────────
from src.experiments import run_rsa
from src.experiments.exp_rsa import RSAResult

if LOAD_FROM_RESULTS and "1_rsa" in experiments_data:
    rsa_d = experiments_data["1_rsa"]
    rsa_result = RSAResult(
        rdm_weird=np.array(rsa_d["rdm_weird"]),
        rdm_sinic=np.array(rsa_d["rdm_sinic"]),
        spearman_r=rsa_d["spearman_r"],
        p_value=rsa_d["p_value"],
        n_permutations=rsa_d["n_permutations"],
        n_pairs=rsa_d["n_pairs"],
        labels=rsa_d["labels"],
    )
    print("RSA result caricato da JSON.")
else:
    rsa_cfg = config.experiments.rsa
    rsa_result = run_rsa(
        emb_weird_core, emb_sinic_core,
        labels=en_core,
        n_permutations=rsa_cfg.n_permutations,
        seed=config.random_seed,
    )

print(f"\nRSA Results:")
print(f"  Spearman r  = {rsa_result.spearman_r:.4f}")
print(f"  Mantel p    = {rsa_result.p_value:.4f}")
print(f"  N pairs     = {rsa_result.n_pairs}")

In [ ]:
# ── Visualizzazione RSA ──────────────────────────────────────────────
from src.visualization.png import (
    plot_clustered_heatmap,
    plot_rdm_correlation,
)

# Heatmap RDM side-by-side con clustering gerarchico
fig_hm = plot_clustered_heatmap(
    rsa_result.rdm_weird, rsa_result.rdm_sinic, rsa_result.labels,
    rsa_result.spearman_r, rsa_result.p_value,
    output_dir=None, domains=domains_core,
)
plt.show()

# Hexbin: correlazione tra distanze RDM
fig_hex = plot_rdm_correlation(
    rsa_result.rdm_weird, rsa_result.rdm_sinic,
    rsa_result.spearman_r, rsa_result.p_value,
    output_dir=None,
)
plt.show()

## Esperimento 2: Gromov-Wasserstein Distance

**Optimal Transport** (Alvarez-Melis & Jaakkola, 2018):
calcola il costo minimo per trasportare la distribuzione di massa dallo
spazio WEIRD allo spazio Sinico preservando le distanze interne.

A differenza dell'RSA (che correla distanze), GW cerca il piano di trasporto
ottimale *senza richiedere che i due spazi vivano nella stessa metrica*.
Una distanza GW bassa indica isomorfismo strutturale.

In [ ]:
# ── Esperimento 2: Gromov-Wasserstein ────────────────────────────────
from src.experiments import gromov_wasserstein_distance
from src.experiments.exp_gw import GWResult

if LOAD_FROM_RESULTS and "2_gromov_wasserstein" in experiments_data:
    gw_d = experiments_data["2_gromov_wasserstein"]
    gw_result = GWResult(
        distance=gw_d["distance"],
        transport_plan=np.array(gw_d["transport_plan"]),
        p_value=gw_d["p_value"],
        n_permutations=gw_d["n_permutations"],
    )
    print("GW result caricato da JSON.")
else:
    gw_cfg = config.experiments.gromov_wasserstein
    gw_result = gromov_wasserstein_distance(
        emb_weird_core, emb_sinic_core,
        entropic_reg=gw_cfg.entropic_reg,
        use_sinkhorn=gw_cfg.use_sinkhorn,
        n_permutations=gw_cfg.n_permutations,
        seed=config.random_seed,
    )

interp = "Alto anisomorfismo" if gw_result.distance > 0.1 else "Relativo isomorfismo"
print(f"\nGromov-Wasserstein Results:")
print(f"  Distanza GW = {gw_result.distance:.6f}")
print(f"  p-value     = {gw_result.p_value:.4f}")
print(f"  Interpretazione: {interp}")

In [ ]:
# ── Visualizzazione GW ───────────────────────────────────────────────
from src.visualization.png import plot_transport_histogram, plot_top_alignments

# Istogramma distribuzione massa del piano di trasporto
fig_th = plot_transport_histogram(
    gw_result.transport_plan, gw_result.distance, gw_result.p_value,
    output_dir=None,
)
plt.show()

# Top-K allineamenti (coppie con massa di trasporto piu' alta)
if rsa_result.labels:
    fig_ta = plot_top_alignments(
        gw_result.transport_plan, rsa_result.labels,
        output_dir=None,
    )
    plt.show()

## Esperimento 3: Assi Assiologici (Kozlowski)

**Axis Projection** (Kozlowski et al., 2019):
proietta i termini giuridici su assi semantici costruiti da coppie di
concetti antitetici (es. individuo/collettivo, positivo/negativo).

Per ogni asse si calcola la correlazione di Spearman tra le proiezioni
nello spazio WEIRD e Sinico, con intervallo di confidenza bootstrap.
Una correlazione alta indica che l'asse cattura la stessa dimensione
semantica in entrambe le tradizioni.

In [ ]:
# ── Esperimento 3: Assi Assiologici ──────────────────────────────────
from src.experiments import run_axes_experiment
from src.experiments.exp_axes import AxesExperimentResult, AxesComparisonResult
from src.experiments.statistical import BootstrapCIResult

if LOAD_FROM_RESULTS and "3_axes" in experiments_data:
    axes_d = experiments_data["3_axes"]
    axes_list = []
    for ax in axes_d["axes"]:
        ci = None
        if ax.get("bootstrap_ci"):
            ci = BootstrapCIResult(
                estimate=ax["bootstrap_ci"].get("estimate", ax["spearman_r"]),
                ci_lower=ax["bootstrap_ci"]["ci_lower"],
                ci_upper=ax["bootstrap_ci"]["ci_upper"],
                n_bootstrap=ax["bootstrap_ci"]["n_bootstrap"],
                alpha=ax["bootstrap_ci"].get("alpha", 0.05),
            )
        axes_list.append(AxesComparisonResult(
            axis_name=ax["axis_name"],
            weird_scores=ax.get("weird_scores", {}),
            sinic_scores=ax.get("sinic_scores", {}),
            spearman_r=ax["spearman_r"],
            spearman_p=ax["spearman_p"],
            bootstrap_ci=ci,
        ))
    axes_result = AxesExperimentResult(
        axes=axes_list,
        labels=list(axes_list[0].weird_scores.keys()) if axes_list else [],
    )
    print("Axes result caricato da JSON.")
else:
    axes_cfg = config.experiments.axes
    axes_result = run_axes_experiment(
        emb_weird_core, emb_sinic_core,
        labels=en_core,
        value_axes=value_axes,
        embed_fn_weird=embed_weird,
        embed_fn_sinic=embed_sinic,
        n_bootstrap=axes_cfg.n_bootstrap,
        seed=config.random_seed,
    )

print(f"\nAxiological Axes Results:")
for ax in axes_result.axes:
    ci_str = ""
    if ax.bootstrap_ci:
        ci_str = f" CI=[{ax.bootstrap_ci.ci_lower:.3f}, {ax.bootstrap_ci.ci_upper:.3f}]"
    print(f"  {ax.axis_name}: rho={ax.spearman_r:.4f}, p={ax.spearman_p:.4f}{ci_str}")

In [ ]:
# ── Visualizzazione Assi ─────────────────────────────────────────────
from src.visualization.png import plot_axes_scatter_ci, plot_forest_plot

# Scatter per ogni asse con etichette outlier
for ax in axes_result.axes:
    ci = ax.bootstrap_ci
    fig_ax = plot_axes_scatter_ci(
        ax.weird_scores, ax.sinic_scores,
        ax.axis_name, ax.spearman_r, ax.spearman_p,
        ci.ci_lower if ci else None,
        ci.ci_upper if ci else None,
        output_dir=None,
    )
    plt.show()

# Forest plot: rho +/- CI per tutti gli assi
axes_for_forest = [
    {
        "axis_name": ax.axis_name,
        "spearman_r": ax.spearman_r,
        "spearman_p": ax.spearman_p,
        "ci_lower": ax.bootstrap_ci.ci_lower if ax.bootstrap_ci else None,
        "ci_upper": ax.bootstrap_ci.ci_upper if ax.bootstrap_ci else None,
    }
    for ax in axes_result.axes
]
fig_fp = plot_forest_plot(axes_for_forest, output_dir=None)
plt.show()

## Esperimento 4: Clustering Gerarchico + Fowlkes-Mallows

**Hierarchical Clustering** con l'indice **Fowlkes-Mallows** (1983):
confronta le strutture gerarchiche dei due spazi applicando clustering
agglomerativo (Ward) e misurando la concordanza tra le partizioni
a diversi livelli di granularita' (multi-k).

FM = 1 indica partizioni identiche; FM = 0 indica assenza di concordanza.

In [ ]:
# ── Esperimento 4: Clustering ────────────────────────────────────────
from src.experiments import run_clustering_experiment
from src.experiments.exp_clustering import (
    ClusteringResult, ClusteringExperimentResult, FMResult,
)

if LOAD_FROM_RESULTS and "4_clustering" in experiments_data:
    clust_d = experiments_data["4_clustering"]
    fm_results = [
        FMResult(
            k=fm["k"],
            fm_index=fm["fm_index"],
            p_value=fm["p_value"],
            n_permutations=fm.get("n_permutations", 5000),
        )
        for fm in clust_d["fm_results"]
    ]
    clustering_result = ClusteringExperimentResult(
        clustering_weird=ClusteringResult(
            linkage_matrix=np.array(clust_d["linkage_weird"]),
            labels=clust_d["labels"],
        ),
        clustering_sinic=ClusteringResult(
            linkage_matrix=np.array(clust_d["linkage_sinic"]),
            labels=clust_d["labels"],
        ),
        fm_results=fm_results,
        labels=clust_d["labels"],
    )
    print("Clustering result caricato da JSON.")
else:
    clust_cfg = config.experiments.clustering
    clustering_result = run_clustering_experiment(
        emb_weird_core, emb_sinic_core,
        labels=en_core,
        method=clust_cfg.method,
        k_values=clust_cfg.k_values,
        n_permutations=clust_cfg.n_permutations,
        seed=config.random_seed,
    )

print(f"\nFowlkes-Mallows Index (multi-k):")
for fm in clustering_result.fm_results:
    interp = "Simile" if fm.fm_index >= 0.5 else "Divergente"
    print(f"  k={fm.k:2d}: FM={fm.fm_index:.4f}, p={fm.p_value:.4f} ({interp})")

In [ ]:
# ── Visualizzazione Clustering ───────────────────────────────────────
from src.visualization.png import plot_truncated_dendrogram, plot_fm_chart

# Dendrogrammi troncati (p=30) side-by-side
fig_dend = plot_truncated_dendrogram(
    clustering_result.clustering_weird.linkage_matrix,
    clustering_result.clustering_sinic.linkage_matrix,
    clustering_result.labels,
    [fm.__dict__ for fm in clustering_result.fm_results],
    output_dir=None,
)
plt.show()

# FM bar chart
fig_fm = plot_fm_chart(
    [fm.__dict__ for fm in clustering_result.fm_results],
    output_dir=None,
)
plt.show()

## Esperimento 5: Neighborhood Divergence Analysis (NDA)

### Part A: Confronto vicinati k-NN
Per ogni termine giuridico, si trovano i k vicini piu' prossimi in entrambi
gli spazi e si calcola la **Jaccard similarity** tra i due insiemi di vicini.
Termini con Jaccard bassa sono *falsi amici*: la stessa parola evoca
associazioni semantiche diverse nelle due tradizioni.

### Part B: Decomposizioni normative
Aritmetica vettoriale (Mikolov et al., 2013) applicata a ipotesi
giurisprudenziali. Es.: *Legge - Stato = ?* — nel mondo WEIRD potrebbe
dare "Giustizia" (legge naturale), nel mondo Sinico "Vuoto" (la legge
e' strumento statale).

In [ ]:
# ── Esperimento 5A: NDA Part A (k-NN Neighborhoods) ──────────────────
from src.experiments import run_nda_part_a, NDAExperimentResult
from src.experiments.exp_nda import (
    NDAPartAResult, NDAPartBResult, TermNeighborhoodResult, DecompositionResult,
)

if LOAD_FROM_RESULTS and "5_nda" in experiments_data:
    nda_d = experiments_data["5_nda"]
    part_a_data = nda_d.get("part_a_neighborhoods", nda_d.get("part_a", {}))
    term_results = [
        TermNeighborhoodResult(
            term_id=tr.get("term", tr.get("label", "")),
            label=tr.get("term", tr.get("label", "")),
            jaccard=tr["jaccard"],
            weird_neighbors=tr["weird_neighbors"],
            sinic_neighbors=tr["sinic_neighbors"],
            shared_neighbors=tr.get("shared_neighbors", []),
        )
        for tr in part_a_data.get("per_term", [])
    ]
    nda_part_a = NDAPartAResult(
        term_results=term_results,
        mean_jaccard=part_a_data["mean_jaccard"],
        p_value=part_a_data["p_value"],
        n_permutations=part_a_data.get("n_permutations", 5000),
        k=part_a_data["k"],
    )
    print("NDA Part A caricato da JSON.")
else:
    nda_cfg = config.experiments.nda
    nda_part_a = run_nda_part_a(
        emb_weird_core, emb_sinic_core,
        core_labels=en_core,
        emb_weird_all=emb_weird_all,
        emb_sinic_all=emb_sinic_all,
        all_labels=en_all,
        k=nda_cfg.k,
        n_permutations=nda_cfg.n_permutations,
        seed=config.random_seed,
    )

print(f"\nNDA Part A:")
print(f"  Mean Jaccard = {nda_part_a.mean_jaccard:.4f}")
print(f"  p-value      = {nda_part_a.p_value:.4f}")
print(f"  k            = {nda_part_a.k}")
print(f"\n  Top 10 False Friends (Jaccard piu' bassa):")
sorted_terms = sorted(nda_part_a.term_results, key=lambda r: r.jaccard)
for r in sorted_terms[:10]:
    print(f"    {r.label:<30} Jaccard={r.jaccard:.3f}")

In [ ]:
# ── Esperimento 5B: NDA Part B (Decomposizioni Normative) ────────────
from src.experiments import run_nda_part_b

if LOAD_FROM_RESULTS and "5_nda" in experiments_data:
    nda_d = experiments_data["5_nda"]
    part_b_data = nda_d.get("part_b_decompositions", nda_d.get("part_b", {}))
    decomp_results = [
        DecompositionResult(
            decomposition_id=d.get("id", ""),
            operation=d.get("operation", "subtraction"),
            en_formula=d["en_formula"],
            zh_formula=d["zh_formula"],
            jurisprudential_question=d.get("jurisprudential_question", ""),
            jaccard=d["jaccard"],
            weird_neighbors=[(n["label"], n.get("cosine_distance", 0)) for n in d["weird_neighbors"]],
            sinic_neighbors=[(n["label"], n.get("cosine_distance", 0)) for n in d["sinic_neighbors"]],
        )
        for d in part_b_data.get("decompositions", [])
    ]
    nda_part_b = NDAPartBResult(decompositions=decomp_results)
    print("NDA Part B caricato da JSON.")
else:
    nda_part_b = run_nda_part_b(
        decompositions,
        embed_fn_weird=embed_weird,
        embed_fn_sinic=embed_sinic,
        corpus_weird=emb_weird_all,
        corpus_sinic=emb_sinic_all,
        corpus_labels=en_all,
        k=config.experiments.nda.k,
    )

# Combina Part A e Part B
nda_result = NDAExperimentResult(part_a=nda_part_a, part_b=nda_part_b)

print(f"\nNDA Part B — Decomposizioni Normative:")
for d in nda_part_b.decompositions:
    print(f"  {d.en_formula:<30} Jaccard={d.jaccard:.3f}")
    print(f"    WEIRD neighbors: {', '.join(l for l, _ in d.weird_neighbors[:5])}")
    print(f"    Sinic neighbors: {', '.join(l for l, _ in d.sinic_neighbors[:5])}")

In [ ]:
# ── Visualizzazione NDA ──────────────────────────────────────────────
from src.visualization.png import (
    plot_jaccard_histogram,
    plot_false_friends_network,
    plot_decomposition_comparison,
)

# Istogramma distribuzione Jaccard
part_a_dict = nda_part_a.to_dict()
fig_jh = plot_jaccard_histogram(
    part_a_dict["per_term"],
    nda_part_a.mean_jaccard,
    nda_part_a.p_value,
    nda_part_a.k,
    output_dir=None,
)
plt.show()

# Network graph dei false friends
fig_ff = plot_false_friends_network(
    part_a_dict["per_term"],
    output_dir=None,
)
plt.show()

# Confronto decomposizioni
if nda_part_b.decompositions:
    part_b_dict = nda_part_b.to_dict()
    fig_dc = plot_decomposition_comparison(
        part_b_dict["decompositions"],
        output_dir=None,
    )
    plt.show()

In [ ]:
# ── Tabella False Friends Top-15 ─────────────────────────────────────

sorted_ff = sorted(nda_part_a.term_results, key=lambda r: r.jaccard)
df_ff = pd.DataFrame([
    {
        "Termine": r.label,
        "Jaccard": f"{r.jaccard:.3f}",
        "Vicini WEIRD (top-3)": ", ".join(r.weird_neighbors[:3]),
        "Vicini Sinici (top-3)": ", ".join(r.sinic_neighbors[:3]),
        "Condivisi": len(r.shared_neighbors),
    }
    for r in sorted_ff[:15]
])
print("\nTop 15 False Friends:")
display(df_ff)

## Supplementare: Visualizzazione UMAP

**UMAP** (McInnes et al., 2018) riduce gli embedding ad alta dimensione a 2D
per visualizzazione. Le coordinate UMAP *non* preservano le distanze assolute,
ma mantengono la struttura locale (vicinato) e globale (cluster).

I due modelli vengono proiettati nello stesso spazio UMAP per confronto visivo.

In [ ]:
# ── UMAP ─────────────────────────────────────────────────────────────
from src.experiments import umap_reduce
from src.experiments.module_c_umap import UMAPResult

if LOAD_FROM_RESULTS and "supplementary_umap" in experiments_data:
    umap_data = experiments_data["supplementary_umap"]
    coords = umap_data["coordinates"]
    weird_coords = coords["weird"]
    sinic_coords = coords["sinic"]
    n_w = len(weird_coords)
    n_s = len(sinic_coords)
    coords_2d = np.zeros((n_w + n_s, 2))
    term_labels_umap = []
    model_labels = np.zeros(n_w + n_s, dtype=int)
    for i, pt in enumerate(weird_coords):
        coords_2d[i, 0] = pt["x"]
        coords_2d[i, 1] = pt["y"]
        term_labels_umap.append(pt["label"])
        model_labels[i] = 0
    for i, pt in enumerate(sinic_coords):
        coords_2d[n_w + i, 0] = pt["x"]
        coords_2d[n_w + i, 1] = pt["y"]
        term_labels_umap.append(pt["label"])
        model_labels[n_w + i] = 1
    umap_result = UMAPResult(
        coords_2d=coords_2d,
        model_labels=model_labels,
        term_labels=term_labels_umap,
    )
    coords_weird = coords_2d[:n_w]
    coords_sinic = coords_2d[n_w:]
    umap_labels = [pt["label"] for pt in weird_coords]
    print("UMAP result caricato da JSON.")
else:
    umap_cfg = config.experiments.umap
    umap_result = umap_reduce(
        emb_weird_core, emb_sinic_core,
        en_core, en_core,
        n_neighbors=umap_cfg.n_neighbors,
        min_dist=umap_cfg.min_dist,
        metric=umap_cfg.metric,
        random_state=config.random_seed,
    )
    n_w = emb_weird_core.shape[0]
    coords_weird = umap_result.coords_2d[:n_w]
    coords_sinic = umap_result.coords_2d[n_w:]
    umap_labels = en_core

print(f"UMAP: {umap_result.coords_2d.shape[0]} punti in 2D")

In [ ]:
# ── Visualizzazione UMAP (matplotlib) ────────────────────────────────
from src.visualization.png import plot_umap_smart_labels

fig_umap = plot_umap_smart_labels(
    coords_weird, coords_sinic, umap_labels,
    output_dir=None, domains=domains_core,
)
plt.show()

In [ ]:
# ── Visualizzazione UMAP (Plotly interattiva) ────────────────────────

fig_plotly = go.Figure()

fig_plotly.add_trace(go.Scatter(
    x=coords_weird[:, 0], y=coords_weird[:, 1],
    mode="markers",
    name="WEIRD (EN)",
    text=umap_labels,
    hovertemplate="<b>%{text}</b><br>x=%{x:.2f}<br>y=%{y:.2f}<extra>WEIRD</extra>",
    marker=dict(size=6, color="#0072B2", opacity=0.7),
))

fig_plotly.add_trace(go.Scatter(
    x=coords_sinic[:, 0], y=coords_sinic[:, 1],
    mode="markers",
    name="Sinic (ZH)",
    text=umap_labels,
    hovertemplate="<b>%{text}</b><br>x=%{x:.2f}<br>y=%{y:.2f}<extra>Sinic</extra>",
    marker=dict(size=6, color="#D55E00", opacity=0.7),
))

fig_plotly.update_layout(
    title="UMAP — Confronto spazi WEIRD vs Sinico",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    width=900, height=600,
    template="plotly_white",
    legend=dict(x=0.01, y=0.99),
)

fig_plotly.show()

In [ ]:
# ── Riepilogo testuale ───────────────────────────────────────────────

print("=" * 70)
print("  RIEPILOGO RISULTATI")
print("=" * 70)
print(f"\n  1. RSA + Mantel")
print(f"     Spearman r = {rsa_result.spearman_r:.4f}, p = {rsa_result.p_value:.4f}")
rsa_interp = "significativa" if rsa_result.p_value < 0.05 else "non significativa"
print(f"     Correlazione tra geometrie: {rsa_interp}")

print(f"\n  2. Gromov-Wasserstein")
print(f"     Distanza = {gw_result.distance:.6f}, p = {gw_result.p_value:.4f}")
gw_interp = "anisomorfismo significativo" if gw_result.distance > 0.1 else "relativo isomorfismo"
print(f"     {gw_interp}")

print(f"\n  3. Assi Assiologici")
for ax in axes_result.axes:
    sig = "*" if ax.spearman_p < 0.05 else ""
    print(f"     {ax.axis_name}: rho={ax.spearman_r:.4f}{sig}")

print(f"\n  4. Clustering (Fowlkes-Mallows)")
for fm in clustering_result.fm_results:
    sig = "*" if fm.p_value < 0.05 else ""
    print(f"     k={fm.k:2d}: FM={fm.fm_index:.4f}{sig}")

print(f"\n  5. NDA")
print(f"     Part A: Mean Jaccard = {nda_part_a.mean_jaccard:.4f}, p = {nda_part_a.p_value:.4f}")
print(f"     Part B: {len(nda_part_b.decompositions)} decomposizioni analizzate")

print(f"\n  Supplementare: UMAP ({umap_result.coords_2d.shape[0]} punti)")
print("=" * 70)
print("  (* = p < 0.05)")

In [ ]:
# ── Tabella riepilogativa + salvataggio ──────────────────────────────

# Tabella riepilogativa
summary_rows = [
    {"Esperimento": "RSA + Mantel", "Statistica": f"r={rsa_result.spearman_r:.4f}",
     "p-value": f"{rsa_result.p_value:.4f}", "Significativo": rsa_result.p_value < 0.05},
    {"Esperimento": "Gromov-Wasserstein", "Statistica": f"d={gw_result.distance:.6f}",
     "p-value": f"{gw_result.p_value:.4f}", "Significativo": gw_result.p_value < 0.05},
]

for ax in axes_result.axes:
    summary_rows.append({
        "Esperimento": f"Asse: {ax.axis_name}",
        "Statistica": f"rho={ax.spearman_r:.4f}",
        "p-value": f"{ax.spearman_p:.4f}",
        "Significativo": ax.spearman_p < 0.05,
    })

for fm in clustering_result.fm_results:
    summary_rows.append({
        "Esperimento": f"Clustering k={fm.k}",
        "Statistica": f"FM={fm.fm_index:.4f}",
        "p-value": f"{fm.p_value:.4f}",
        "Significativo": fm.p_value < 0.05,
    })

summary_rows.append({
    "Esperimento": "NDA (Mean Jaccard)",
    "Statistica": f"J={nda_part_a.mean_jaccard:.4f}",
    "p-value": f"{nda_part_a.p_value:.4f}",
    "Significativo": nda_part_a.p_value < 0.05,
})

df_summary = pd.DataFrame(summary_rows)
print("Tabella riepilogativa:")
display(df_summary)

# Salvataggio results.json (solo se pipeline completa)
if not LOAD_FROM_RESULTS:
    from src.core.output_manager import OutputManager
    from src.core.hashing import compute_config_hash, compute_file_hash

    data_path = config.get_absolute_path("processed") / "legal_terms.json"
    input_data_hash = compute_file_hash(data_path)
    config_hash = compute_config_hash(config)

    output_manager = OutputManager(config, input_data_hash, config_hash)
    output_manager.set_rsa_result(rsa_result)
    output_manager.set_gw_result(gw_result)
    output_manager.set_axes_result(axes_result)
    output_manager.set_clustering_result(clustering_result)
    output_manager.set_nda_result(nda_result)
    output_manager.set_umap_result(coords_weird, coords_sinic, en_core, en_core)

    results_path = output_manager.save()
    print(f"\nRisultati salvati in: {results_path}")
else:
    print("\nModalita' LOAD_FROM_RESULTS: nessun salvataggio.")